# Echo Chat Agent - Hosted Agent 배포

LLM 호출 없이 입력을 그대로 Echo하는 더미 에이전트입니다.  
Hosted Agent 인터페이스(Responses API)가 정상 동작하는지 테스트하는 용도입니다.

---

## 0. 환경 설정

In [1]:
%pip install --quiet python-dotenv "azure-ai-projects>=2.0.0" azure-identity


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import json
from dotenv import load_dotenv

load_dotenv()

SUBSCRIPTION_ID = os.getenv("SUBSCRIPTION_ID")
RESOURCE_GROUP = os.getenv("RESOURCE_GROUP")
ACCOUNT_NAME = os.getenv("ACCOUNT_NAME")
ACR_NAME = os.getenv("ACR_NAME")
AZURE_AI_PROJECT_ENDPOINT = os.getenv("AZURE_AI_PROJECT_ENDPOINT")

print(f"Subscription: {SUBSCRIPTION_ID[:8]}...")
print(f"Resource Group: {RESOURCE_GROUP}")
print(f"Account Name: {ACCOUNT_NAME}")
print(f"Project Endpoint: {AZURE_AI_PROJECT_ENDPOINT[:20]}...")
print(f"ACR: {ACR_NAME}")

Subscription: c0a6269b...
Resource Group: rg-ms-foundry
Account Name: ms-foundry-1023
Project Endpoint: https://ms-foundry-1...
ACR: agentregistry1023


## 1. Docker 이미지 빌드 & 푸시

> ⚠️ Apple Silicon 맥북에서는 `--platform=linux/amd64` 필수

In [9]:
AGENT_NAME = "echo-chat"
IMAGE_NAME = "echo-chat"
IMAGE_TAG = "v1"
FULL_IMAGE = f"{ACR_NAME}.azurecr.io/{IMAGE_NAME}:{IMAGE_TAG}"

print("터미널에서 아래 명령을 실행하세요:\n")
print(f"az acr login --name {ACR_NAME}")
print(f"docker build --platform=linux/amd64 -t {FULL_IMAGE} .")
print(f"docker push {FULL_IMAGE}")

터미널에서 아래 명령을 실행하세요:

az acr login --name agentregistry1023
docker build --platform=linux/amd64 -t agentregistry1023.azurecr.io/echo-chat:v1 .
docker push agentregistry1023.azurecr.io/echo-chat:v1


## 2. Hosted Agent 생성

In [ ]:
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    HostedAgentDefinition,
    ProtocolVersionRecord,
    AgentProtocol,
)
from azure.identity import DefaultAzureCredential

project = AIProjectClient(
    endpoint=AZURE_AI_PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
    allow_preview=True,
)

agent = project.agents.create_version(
    agent_name=AGENT_NAME,
    description="Dummy echo agent for testing Hosted Agent interface",
    definition=HostedAgentDefinition(
        container_protocol_versions=[
            ProtocolVersionRecord(protocol=AgentProtocol.RESPONSES, version="v12)
        ],
        cpu="1",
        memory="2Gi",
        image=FULL_IMAGE,
    ),
)

print(f"✅ Agent created: {agent.name}, version: {agent.version}")

✅ Agent created: echo-chat, version: 1


## 3. Agent 테스트

In [5]:
agent_info = project.agents.get(agent_name=AGENT_NAME)
print(f"Agent: {agent_info.name}")
print(f"Versions: {agent_info.versions}")

Agent: echo-chat
Versions: {'latest': {'metadata': {}, 'object': 'agent.version', 'id': 'echo-chat:1', 'name': 'echo-chat', 'version': '1', 'description': 'Dummy echo agent for testing Hosted Agent interface', 'created_at': 1776326219, 'definition': {'kind': 'hosted', 'container_protocol_versions': [{'protocol': 'responses', 'version': 'v1'}], 'cpu': '1', 'memory': '2Gi', 'image': 'agentregistry1023.azurecr.io/echo-chat:v1'}, 'status': 'active'}}


In [7]:
openai_client = project.get_openai_client()

stream = openai_client.responses.create(
    stream=True,
    input=[{"role": "user", "content": "hello echo test"}],
    extra_body={"agent_reference": {"name": agent_info.name, "type": "agent_reference"}},
)

for event in stream:
    if event.type == "response.output_text.delta":
        print(event.delta, end="", flush=True)
print()

## 4. 정리

In [ ]:
project.agents.delete_version(agent_name=AGENT_NAME, agent_version=agent.version)
print(f"Agent version {agent.version} deleted.")